In [1]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# Define SKUs and their respective base demand & start dates (matching your laptop_specs.csv)
sku_configs = {
    'MBA13': {'base_demand': 280, 'start_date': '2022-01-01', 'end_date': '2025-12-31'},
    'MBA15': {'base_demand': 240, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
    'MBP14': {'base_demand': 180, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
    'MBP16': {'base_demand': 110, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
}

all_records = []

for sku, config in sku_configs.items():
    date_range = pd.date_range(start=config['start_date'], end=config['end_date'], freq='D')
    base_demand = config['base_demand']
    
    for dt in date_range:
        # 1. Base trend (slight annual growth)
        year_offset = (dt.year - 2022) * 0.03  # 3% growth per year
        
        # 2. Seasonality Multipliers
        # Day of week: Weekdays sell slightly better than weekends
        day_mult = 0.85 if dt.dayofweek in [5, 6] else 1.05
        
        # Back to School (Aug-Sept) & Holiday Season (Nov-Dec)
        season_mult = 1.0
        if dt.month in [8, 9]:
            season_mult = 1.35  # +35% Back-to-school boost
        elif dt.month in [11, 12]:
            season_mult = 1.45  # +45% Holiday boost
            
        # 3. Calculate expected demand
        expected_demand = base_demand * (1 + year_offset) * day_mult * season_mult
        
        # Add Poisson / Normal random noise
        demand = np.random.normal(loc=expected_demand, scale=base_demand * 0.12)
        
        # 4. Inject Non-Linear Shocks (5% chance of promo spike, 3% chance of supply stockout)
        shock_roll = np.random.rand()
        if shock_roll < 0.05:
            demand *= np.random.uniform(1.30, 1.60)  # +30% to +60% Promotional Spike
        elif shock_roll > 0.97:
            demand *= np.random.uniform(0.30, 0.60)  # -40% to -70% Supply Chain Stockout
            
        # Ensure units_sold is an integer and non-negative
        units_sold = int(max(0, round(demand)))
        
        all_records.append({
            'date': dt.strftime('%Y-%m-%d'),
            'sku_id': sku,
            'units_sold': units_sold
        })

# Create DataFrame
new_demand_df = pd.DataFrame(all_records)

# Sort chronologically by sku_id and date
new_demand_df['date'] = pd.to_datetime(new_demand_df['date'])
new_demand_df = new_demand_df.sort_values(['sku_id', 'date']).reset_index(drop=True)

# Format date back to string YYYY-MM-DD
new_demand_df['date'] = new_demand_df['date'].dt.strftime('%Y-%m-%d')

# Preview
print("Dataset Shape:", new_demand_df.shape)
print("\nFirst 5 rows:")
print(new_demand_df.head())

print("\nSummary Statistics per SKU:")
print(new_demand_df.groupby('sku_id')['units_sold'].describe())

# Save to data/raw/demand_timeseries_v2.csv WITHOUT touching the original file
output_path = "../data/raw/demand_timeseries_v2.csv"
new_demand_df.to_csv(output_path, index=False)
print(f"\nSaved new dataset successfully to: {output_path}")

Dataset Shape: (4749, 3)

First 5 rows:
         date sku_id  units_sold
0  2022-01-01  MBA13         255
1  2022-01-02  MBA13         233
2  2022-01-03  MBA13         286
3  2022-01-04  MBA13         286
4  2022-01-05  MBA13         552

Summary Statistics per SKU:
         count        mean        std   min     25%    50%     75%    max
sku_id                                                                   
MBA13   1461.0  334.520192  87.656016  77.0  279.00  321.0  394.00  804.0
MBA15   1096.0  285.621350  79.006004  85.0  238.75  275.0  335.25  662.0
MBP14   1096.0  216.178832  57.161479  44.0  179.00  205.0  255.25  479.0
MBP16   1096.0  132.960766  34.838903  36.0  111.00  128.0  154.00  306.0

Saved new dataset successfully to: ../data/raw/demand_timeseries_v2.csv


In [1]:
import numpy as np
import pandas as pd

# 1. GENERATE V3 DATASET (CONDITIONAL SHOCKS)
np.random.seed(42)

sku_configs_v3 = {
    'MBA13': {'base_demand': 280, 'start_date': '2022-01-01', 'end_date': '2025-12-31'},
    'MBA15': {'base_demand': 240, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
    'MBP14': {'base_demand': 180, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
    'MBP16': {'base_demand': 110, 'start_date': '2023-01-01', 'end_date': '2025-12-31'},
}

all_records_v3 = []

for sku_v3, config_v3 in sku_configs_v3.items():
    date_range_v3 = pd.date_range(start=config_v3['start_date'], end=config_v3['end_date'], freq='D')
    base_demand_v3 = config_v3['base_demand']
    
    for dt_v3 in date_range_v3:
        year_offset_v3 = (dt_v3.year - 2022) * 0.03
        day_mult_v3 = 0.85 if dt_v3.dayofweek in [5, 6] else 1.05
        
        season_mult_v3 = 1.0
        if dt_v3.month in [8, 9]: season_mult_v3 = 1.35
        elif dt_v3.month in [11, 12]: season_mult_v3 = 1.45
            
        expected_demand_v3 = base_demand_v3 * (1 + year_offset_v3) * day_mult_v3 * season_mult_v3
        
        is_promo_v3 = 0
        is_stockout_v3 = 0
        shock_roll_v3 = np.random.rand()
        
        # CONDITIONAL SHOCKS
        if shock_roll_v3 < 0.10:
            is_promo_v3 = 1
            if dt_v3.dayofweek in [5, 6]:
                expected_demand_v3 *= 2.50 # Massive weekend spike
            else:
                expected_demand_v3 *= 1.05 # Tiny weekday boost
                
        elif shock_roll_v3 > 0.95:
            is_stockout_v3 = 1
            if dt_v3.month in [11, 12]:
                expected_demand_v3 *= 0.20 # Severe holiday drop
            else:
                expected_demand_v3 *= 0.80 # Mild drop
                
        demand_v3 = np.random.normal(loc=expected_demand_v3, scale=base_demand_v3 * 0.05)
        
        all_records_v3.append({
            'date': dt_v3.strftime('%Y-%m-%d'),
            'sku_id': sku_v3,
            'units_sold': int(max(0, round(demand_v3))),
            'is_promo': is_promo_v3,
            'is_stockout': is_stockout_v3
        })

new_demand_df_v3 = pd.DataFrame(all_records_v3)
new_demand_df_v3['date'] = pd.to_datetime(new_demand_df_v3['date'])
new_demand_df_v3 = new_demand_df_v3.sort_values(['sku_id', 'date']).reset_index(drop=True)
new_demand_df_v3['date'] = new_demand_df_v3['date'].dt.strftime('%Y-%m-%d')

output_path_v3 = "../data/raw/demand_timeseries_v3.csv"
new_demand_df_v3.to_csv(output_path_v3, index=False)
print(f"Saved raw v3 synthetic dataset to: {output_path_v3}")
print(new_demand_df_v3.head())

Saved raw v3 synthetic dataset to: ../data/raw/demand_timeseries_v3.csv
         date sku_id  units_sold  is_promo  is_stockout
0  2022-01-01  MBA13         222         0            0
1  2022-01-02  MBA13         242         0            0
2  2022-01-03  MBA13         313         1            0
3  2022-01-04  MBA13         308         0            0
4  2022-01-05  MBA13         301         1            0


In [1]:
import pandas as pd
import numpy as np
from itertools import product
import warnings
warnings.filterwarnings("ignore")

# 1. SETUP DIMENSIONS (1 Million Rows)
dates = pd.date_range(start="2022-01-01", periods=1000, freq="D")
stores = [f"Store_{i}" for i in range(1, 251)]
skus = ["MBA13", "MBA15", "MBP14", "MBP16"]

# Create Cartesian Product
df_1m = pd.DataFrame(list(product(dates, stores, skus)), columns=["date", "store_id", "sku_id"])

# 2. ADD CALENDAR FEATURES
df_1m['year'] = df_1m['date'].dt.year
df_1m['month'] = df_1m['date'].dt.month
df_1m['day_of_week'] = df_1m['date'].dt.dayofweek
df_1m['is_weekend'] = df_1m['day_of_week'].isin([5, 6]).astype(int)
df_1m['is_holiday_season'] = df_1m['month'].isin([11, 12]).astype(int)
df_1m['is_back_to_school'] = df_1m['month'].isin([8, 9]).astype(int)

# 3. INJECT CONDITIONAL SHOCKS & BASE DEMAND
np.random.seed(42)
df_1m['is_promo'] = np.random.choice([0, 1], size=len(df_1m), p=[0.95, 0.05])
df_1m['is_stockout'] = np.random.choice([0, 1], size=len(df_1m), p=[0.98, 0.02])

# Base demand generation with conditional logic
base_demand = np.random.normal(loc=50, scale=10, size=len(df_1m))
df_1m['units_sold'] = base_demand

# Apply Shocks (Conditional Logic for Tree Models)
promo_multiplier = np.where(df_1m['is_weekend'] == 1, 2.5, 1.05)
stockout_penalty = np.where(df_1m['is_holiday_season'] == 1, 0.2, 0.6)

df_1m['units_sold'] = np.where(df_1m['is_promo'] == 1, df_1m['units_sold'] * promo_multiplier, df_1m['units_sold'])
df_1m['units_sold'] = np.where(df_1m['is_stockout'] == 1, df_1m['units_sold'] * stockout_penalty, df_1m['units_sold'])
df_1m['units_sold'] = np.maximum(0, df_1m['units_sold'].astype(int))

# 4. COMPUTE TIME-SERIES FEATURES (Grouped safely by Store AND SKU)
df_1m = df_1m.sort_values(by=['store_id', 'sku_id', 'date'])

df_1m['lag_1'] = df_1m.groupby(['store_id', 'sku_id'])['units_sold'].shift(1)
df_1m['lag_7'] = df_1m.groupby(['store_id', 'sku_id'])['units_sold'].shift(7)
df_1m['rolling_mean_7'] = df_1m.groupby(['store_id', 'sku_id'])['units_sold'].transform(lambda x: x.shift(1).rolling(7).mean())
df_1m['rolling_mean_30'] = df_1m.groupby(['store_id', 'sku_id'])['units_sold'].transform(lambda x: x.shift(1).rolling(30).mean())

# Detrending Target (Difference from yesterday)
df_1m['units_sold_diff'] = df_1m['units_sold'] - df_1m['lag_1']

# Drop NaNs created by lags
df_1m = df_1m.dropna().reset_index(drop=True)

# Save to disk
df_1m.to_csv("../data/processed/demand_features_v4_1M.csv", index=False)
print(f"Dataset generated successfully with {len(df_1m)} rows.")

Dataset generated successfully with 970000 rows.
